# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [2]:
import pandas as pd

---
## 2. Load data

In [3]:
df = pd.read_csv(r"C:\Users\enock\Documents\Project BCG Group Data Science\clean_data_after_eda.csv")
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [4]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [5]:
price_df = pd.read_csv(r"C:\Users\enock\Documents\Project BCG Group Data Science\price_data.csv")
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [6]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [7]:
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [10]:
price_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193002 entries, 0 to 193001
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   id                  193002 non-null  object        
 1   price_date          193002 non-null  datetime64[ns]
 2   price_off_peak_var  193002 non-null  float64       
 3   price_peak_var      193002 non-null  float64       
 4   price_mid_peak_var  193002 non-null  float64       
 5   price_off_peak_fix  193002 non-null  float64       
 6   price_peak_fix      193002 non-null  float64       
 7   price_mid_peak_fix  193002 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 11.8+ MB


In [12]:
energy_cols = ["price_off_peak_var", "price_mid_peak_var", "price_peak_var"]

agg_dict = {}

for col in energy_cols:
    agg_dict[col] = ["mean", "std", "min", "max", "last"]

price_agg = (price_df.groupby("id").agg(agg_dict))

price_agg.columns = [f"{col}_{stat}" for col, stat in price_agg.columns]

price_agg = price_agg.reset_index()

In [13]:
price_agg

,id,price_off_peak_var_mean,price_off_peak_var_std,price_off_peak_var_min,price_off_peak_var_max,price_off_peak_var_last,price_mid_peak_var_mean,price_mid_peak_var_std,price_mid_peak_var_min,price_mid_peak_var_max,price_mid_peak_var_last,price_peak_var_mean,price_peak_var_std,price_peak_var_min,price_peak_var_max,price_peak_var_last
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.003976,0.119906,0.128067,0.119906,0.073160,0.001368,0.070232,0.073773,0.073719,0.103794,0.001989,0.101673,0.105842,0.101673
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.002197,0.143943,0.148405,0.143943,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.026008,0.150837,0.205742,0.201280,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.005049,0.113068,0.123086,0.113068,0.069032,0.000403,0.068646,0.069463,0.069409,0.098292,0.002580,0.095385,0.100505,0.095385
4,00114d74e963e47177db89bc70108537,0.147926,0.002202,0.145440,0.149902,0.145440,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16091,ffef185810e44254c3a4c6395e6b4d8a,0.138863,0.026238,0.112488,0.165037,0.112488,0.080780,0.012503,0.068829,0.093881,0.068829,0.115125,0.020548,0.094804,0.135909,0.094804
16092,fffac626da707b1b5ab11e8431a4d0a2,0.147137,0.002098,0.144363,0.148825,0.145047,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16093,fffc0cacd305dd51f316424bbb08d1bd,0.153879,0.003044,0.151399,0.159560,0.151399,0.094842,0.004310,0.091394,0.101037,0.091394,0.129497,0.002447,0.126871,0.132895,0.126871
16094,fffe4f5646aa39c7f97f95ae2679ce64,0.123858,0.004600,0.118175,0.127566,0.118175,0.073735,0.000471,0.073433,0.074516,0.074516,0.103499,0.002397,0.100491,0.105428,0.100491


In [15]:
#4 price change over time (recent vs past)

for col in energy_cols:
    first = price_df.groupby("id")[col].first()
    last = price_df.groupby("id")[col].last()
    price_agg[f"{col}_change_total"] = last - first

In [17]:
price_agg["peak_minus_offpeak_last"] = (
    price_agg["price_peak_var_last"] -
    price_agg["price_off_peak_var_last"]
)

price_agg["mid_minus_offpeak_last"] = (
    price_agg["price_mid_peak_var_last"] -
    price_agg["price_off_peak_var_last"]
)

price_agg["peak_minus_mid_last"] = (
    price_agg["price_peak_var_last"] -
    price_agg["price_mid_peak_var_last"]
)

In [18]:
price_agg

,id,price_off_peak_var_mean,price_off_peak_var_std,price_off_peak_var_min,price_off_peak_var_max,price_off_peak_var_last,price_mid_peak_var_mean,price_mid_peak_var_std,price_mid_peak_var_min,price_mid_peak_var_max,...,price_peak_var_std,price_peak_var_min,price_peak_var_max,price_peak_var_last,price_off_peak_var_change_total,price_mid_peak_var_change_total,price_peak_var_change_total,peak_minus_offpeak_last,mid_minus_offpeak_last,peak_minus_mid_last
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.003976,0.119906,0.128067,0.119906,0.073160,0.001368,0.070232,0.073773,...,0.001989,0.101673,0.105842,0.101673,NaN,NaN,NaN,-0.018233,-0.046187,0.027954
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.002197,0.143943,0.148405,0.143943,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,-0.143943,-0.143943,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.026008,0.150837,0.205742,0.201280,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,-0.201280,-0.201280,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.005049,0.113068,0.123086,0.113068,0.069032,0.000403,0.068646,0.069463,...,0.002580,0.095385,0.100505,0.095385,NaN,NaN,NaN,-0.017683,-0.043659,0.025976
4,00114d74e963e47177db89bc70108537,0.147926,0.002202,0.145440,0.149902,0.145440,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,-0.145440,-0.145440,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16091,ffef185810e44254c3a4c6395e6b4d8a,0.138863,0.026238,0.112488,0.165037,0.112488,0.080780,0.012503,0.068829,0.093881,...,0.020548,0.094804,0.135909,0.094804,NaN,NaN,NaN,-0.017684,-0.043659,0.025975
16092,fffac626da707b1b5ab11e8431a4d0a2,0.147137,0.002098,0.144363,0.148825,0.145047,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,-0.145047,-0.145047,0.000000
16093,fffc0cacd305dd51f316424bbb08d1bd,0.153879,0.003044,0.151399,0.159560,0.151399,0.094842,0.004310,0.091394,0.101037,...,0.002447,0.126871,0.132895,0.126871,NaN,NaN,NaN,-0.024528,-0.060005,0.035477
16094,fffe4f5646aa39c7f97f95ae2679ce64,0.123858,0.004600,0.118175,0.127566,0.118175,0.073735,0.000471,0.073433,0.074516,...,0.002397,0.100491,0.105428,0.100491,NaN,NaN,NaN,-0.017684,-0.043659,0.025975


In [20]:
for col in energy_cols:
    price_agg[f"{col}_volatility_ratio"] = (
        price_agg[f"{col}_std"] / price_agg[f"{col}_mean"]
    )

In [21]:
price_agg

,id,price_off_peak_var_mean,price_off_peak_var_std,price_off_peak_var_min,price_off_peak_var_max,price_off_peak_var_last,price_mid_peak_var_mean,price_mid_peak_var_std,price_mid_peak_var_min,price_mid_peak_var_max,...,price_peak_var_last,price_off_peak_var_change_total,price_mid_peak_var_change_total,price_peak_var_change_total,peak_minus_offpeak_last,mid_minus_offpeak_last,peak_minus_mid_last,price_off_peak_var_volatility_ratio,price_mid_peak_var_volatility_ratio,price_peak_var_volatility_ratio
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.003976,0.119906,0.128067,0.119906,0.073160,0.001368,0.070232,0.073773,...,0.101673,NaN,NaN,NaN,-0.018233,-0.046187,0.027954,0.031981,0.018700,0.019166
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.002197,0.143943,0.148405,0.143943,0.000000,0.000000,0.000000,0.000000,...,0.000000,NaN,NaN,NaN,-0.143943,-0.143943,0.000000,0.015003,NaN,NaN
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.026008,0.150837,0.205742,0.201280,0.000000,0.000000,0.000000,0.000000,...,0.000000,NaN,NaN,NaN,-0.201280,-0.201280,0.000000,0.143248,NaN,NaN
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.005049,0.113068,0.123086,0.113068,0.069032,0.000403,0.068646,0.069463,...,0.095385,NaN,NaN,NaN,-0.017683,-0.043659,0.025976,0.042512,0.005844,0.026250
4,00114d74e963e47177db89bc70108537,0.147926,0.002202,0.145440,0.149902,0.145440,0.000000,0.000000,0.000000,0.000000,...,0.000000,NaN,NaN,NaN,-0.145440,-0.145440,0.000000,0.014886,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16091,ffef185810e44254c3a4c6395e6b4d8a,0.138863,0.026238,0.112488,0.165037,0.112488,0.080780,0.012503,0.068829,0.093881,...,0.094804,NaN,NaN,NaN,-0.017684,-0.043659,0.025975,0.188947,0.154773,0.178488
16092,fffac626da707b1b5ab11e8431a4d0a2,0.147137,0.002098,0.144363,0.148825,0.145047,0.000000,0.000000,0.000000,0.000000,...,0.000000,NaN,NaN,NaN,-0.145047,-0.145047,0.000000,0.014262,NaN,NaN
16093,fffc0cacd305dd51f316424bbb08d1bd,0.153879,0.003044,0.151399,0.159560,0.151399,0.094842,0.004310,0.091394,0.101037,...,0.126871,NaN,NaN,NaN,-0.024528,-0.060005,0.035477,0.019785,0.045446,0.018892
16094,fffe4f5646aa39c7f97f95ae2679ce64,0.123858,0.004600,0.118175,0.127566,0.118175,0.073735,0.000471,0.073433,0.074516,...,0.100491,NaN,NaN,NaN,-0.017684,-0.043659,0.025975,0.037142,0.006391,0.023164


In [22]:
price_df = price_df.merge(price_agg, on = "id")

In [23]:
price_df

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix,price_off_peak_var_mean,price_off_peak_var_std,...,price_peak_var_last,price_off_peak_var_change_total,price_mid_peak_var_change_total,price_peak_var_change_total,peak_minus_offpeak_last,mid_minus_offpeak_last,peak_minus_mid_last,price_off_peak_var_volatility_ratio,price_mid_peak_var_volatility_ratio,price_peak_var_volatility_ratio
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.000000,0.000000,44.266931,0.00000,0.000000,0.14855,0.002461,...,0.000000,NaN,NaN,NaN,-0.145859,-0.145859,0.000000,0.016567,NaN,NaN
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.000000,0.000000,44.266931,0.00000,0.000000,0.14855,0.002461,...,0.000000,NaN,NaN,NaN,-0.145859,-0.145859,0.000000,0.016567,NaN,NaN
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.000000,0.000000,44.266931,0.00000,0.000000,0.14855,0.002461,...,0.000000,NaN,NaN,NaN,-0.145859,-0.145859,0.000000,0.016567,NaN,NaN
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.000000,0.000000,44.266931,0.00000,0.000000,0.14855,0.002461,...,0.000000,NaN,NaN,NaN,-0.145859,-0.145859,0.000000,0.016567,NaN,NaN
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.000000,0.000000,44.266931,0.00000,0.000000,0.14855,0.002461,...,0.000000,NaN,NaN,NaN,-0.145859,-0.145859,0.000000,0.016567,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192997,16f51cdc2baa19af0b940ee1b3dd17d5,2015-08-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555,0.12536,0.004821,...,0.102232,NaN,NaN,NaN,-0.017684,-0.043659,0.025975,0.038456,0.008716,0.022443
192998,16f51cdc2baa19af0b940ee1b3dd17d5,2015-09-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555,0.12536,0.004821,...,0.102232,NaN,NaN,NaN,-0.017684,-0.043659,0.025975,0.038456,0.008716,0.022443
192999,16f51cdc2baa19af0b940ee1b3dd17d5,2015-10-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555,0.12536,0.004821,...,0.102232,NaN,NaN,NaN,-0.017684,-0.043659,0.025975,0.038456,0.008716,0.022443
193000,16f51cdc2baa19af0b940ee1b3dd17d5,2015-11-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555,0.12536,0.004821,...,0.102232,NaN,NaN,NaN,-0.017684,-0.043659,0.025975,0.038456,0.008716,0.022443
